# kwargs-pass-through-recipe — ex1: thread kwargs into forward call AND Recipe

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `kwargs-pass-through-recipe`. Running the final beacon cell reports progress against the `Backprop: Kwargs pass-through` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Kwargs pass-through` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kwargs-pass-through-recipe`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kwargs-pass-through-recipe"
DD_SUBTOPIC = "Backprop: Kwargs pass-through"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Kwargs pass-through Recipe — quick refresher

Some forward ops take **keyword args** that change the output (`dim`, `keepdim`, `new_shape`, ...). The autograd wrapper has to thread them in **two** places:

```python
def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw = tuple(a.array if isinstance(a, Tensor) else a for a in args)
        out_raw = fwd_fn(*raw, **kwargs)               # (1) into the call
        out = Tensor(out_raw, requires_grad)
        out.recipe = Recipe(fwd_fn, raw, kwargs, parents)   # (2) into Recipe
        return out
    return tensor_func
```

Why both? **(1)** the forward call needs the kwargs to compute the right output. **(2)** the backward fn needs the *same* kwargs at reverse time — e.g. `sum_back` needs to know which `dim` was summed so it can broadcast back. Drop them from the Recipe and reverse-pass tests start failing on shape mismatches even though the forward looks fine.

### Exercise 1 — thread kwargs into forward call AND Recipe

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the kwargs-pass-through pattern by routing keyword args into both the forward call and the constructed Recipe so a downstream back fn can replay the op.
> Keywords: kwargs, recipe, wrap-forward-fn, sum, dim
> ```

**KCs targeted:** `kwargs-pass-through-recipe`, `recipe-dataclass`

We've given you a stripped-down `Tensor` wrapper and a `Recipe` dataclass. Implement `wrap_forward_fn(fwd_fn)`, which returns a closure `tensor_func(*args, **kwargs)` that:

1. **Unboxes** every Tensor input — pull `.array` out of each, leave non-Tensors alone.
2. Calls `fwd_fn(*raw_args, **kwargs)` to compute the raw output.    **The kwargs must reach the forward call** — otherwise `sum(x, dim=1)` would silently reduce over the wrong axis.
3. Boxes the result in `Tensor(out_raw)` and attaches    `out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)`.    **The same kwargs dict must be stored on the Recipe** so the backward fn can be called with `**recipe.kwargs` at reverse time.
4. Returns the boxed Tensor.

Use `parents = {idx: a for idx, a in enumerate(args) if isinstance(a, Tensor)}`.

**Why kwargs are the failure mode.** It's tempting to write `Recipe(fwd_fn, raw_args, {}, parents)` (empty dict) if your tests don't reach the reverse pass yet — the forward result is correct, so the bug is invisible. Then `sum_back` runs at reverse time, doesn't know which `dim` was reduced, broadcasts wrong, and shape errors blow up far from the cause.

Don't call `torch.autograd`; we're building the autograd layer manually.

In [ ]:
from dataclasses import dataclass
from typing import Callable, Any


@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict


class MiniTensor:
    """Minimal Tensor wrapper around a torch.Tensor (named `array`)."""
    def __init__(self, array):
        self.array = array
        self.recipe = None


def wrap_forward_fn(fwd_fn: Callable) -> Callable:
    """Return tensor_func that boxes/unboxes around fwd_fn and threads kwargs."""
    raise NotImplementedError()


def _test_ex1():
    # --- forward call gets kwargs (sum over dim=1) ---
    wrapped_sum = wrap_forward_fn(t.sum)
    x = MiniTensor(t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]))
    out = wrapped_sum(x, dim=1)
    assert isinstance(out, MiniTensor), 'output must be a MiniTensor'
    assert t.allclose(out.array, t.tensor([6.0, 15.0])), (
        f'forward dim=1 was ignored: out.array={out.array} '
        f'(expected [6, 15])'
    )

    # --- Recipe carries the same kwargs that the call used ---
    assert out.recipe is not None, 'Recipe was never attached'
    assert out.recipe.func is t.sum, 'Recipe.func wrong'
    assert out.recipe.kwargs == {'dim': 1}, (
        f'Recipe.kwargs missing or wrong: {out.recipe.kwargs}'
    )
    assert 0 in out.recipe.parents, 'parents missing arg-0 Tensor'
    assert out.recipe.parents[0] is x, 'parents must reference the original Tensor'

    # --- a SECOND kwarg also threads through (keepdim) ---
    out2 = wrapped_sum(x, dim=1, keepdim=True)
    assert out2.array.shape == (2, 1), (
        f'keepdim=True ignored: out2.shape={out2.array.shape}'
    )
    assert out2.recipe.kwargs == {'dim': 1, 'keepdim': True}, (
        f'Recipe lost keepdim: {out2.recipe.kwargs}'
    )

    # --- no kwargs case still works (empty dict stored) ---
    wrapped_log = wrap_forward_fn(t.log)
    y = MiniTensor(t.tensor([1.0, t.e, t.e * t.e]))
    out3 = wrapped_log(y)
    assert t.allclose(out3.array, t.tensor([0.0, 1.0, 2.0]), atol=1e-5)
    assert out3.recipe.kwargs == {}, (
        f'kwargs should be empty dict, got {out3.recipe.kwargs}'
    )

    # --- args on Recipe are RAW (unboxed) tensors ---
    assert isinstance(out.recipe.args[0], t.Tensor), (
        f'Recipe.args[0] should be raw torch.Tensor, got {type(out.recipe.args[0])}'
    )
    assert not isinstance(out.recipe.args[0], MiniTensor), (
        'Recipe.args should hold the unboxed raw tensor, not the MiniTensor'
    )

    # --- proof: a downstream back fn can REPLAY the op using recipe.kwargs ---
    # (Forward shape-restore: a correct sum_back would broadcast grad back
    # along the same dim — only possible because kwargs are preserved.)
    grad_out = t.tensor([1.0, 1.0])
    replayed = grad_out.unsqueeze(out.recipe.kwargs['dim']).expand_as(x.array)
    assert replayed.shape == x.array.shape, 'shape replay would fail'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def wrap_forward_fn(fwd_fn: Callable) -> Callable:
    def tensor_func(*args, **kwargs):
        # 1. unbox MiniTensor inputs to raw torch.Tensor (pass-through non-Tensors)
        raw_args = tuple(
            a.array if isinstance(a, MiniTensor) else a for a in args
        )
        # 2. forward call MUST receive the kwargs
        out_raw = fwd_fn(*raw_args, **kwargs)
        # 3. box result and attach Recipe — kwargs preserved for reverse pass
        parents = {
            idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)
        }
        out = MiniTensor(out_raw)
        out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func
```

**Two places the kwargs go, ALWAYS in this order.**
1. `fwd_fn(*raw_args, **kwargs)` — without this, the forward output is wrong (e.g. `sum` reduces over the default axis instead of `dim`).
2. `Recipe(..., kwargs, ...)` — without this, the reverse pass has no way to call `back_fn(grad_out, out, *recipe.args, **recipe.kwargs)` with the same kwargs the forward used.

**Why it's a `dict`, not unpacked.** Because the back fn signature is `(grad_out, out, *args, **kwargs)`, the reverse pass dispatches with `back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)`. Storing kwargs as the raw dict means the dispatcher line is generic — no per-op switch needed.

**The silent-bug failure mode.** If you forget step (2) but remember step (1), the forward looks correct and the bug only triggers when the reverse pass runs. The ARENA notebook hints at exactly this trap: 'if you're failing tests but think your implementation is correct, go back and check this.'
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()